In [ ]:
import pandas as pd
import numpy as np
import os

# project root
os.chdir('c:/dev/AnalystLab_Internship')

# Now use relative path from root
netflix_path = "Week_1/data/netflix_titles.csv"
retail_path = "Week_1/data/online_retail/OnlineRetail.csv"

netflix_df = pd.read_csv(netflix_path)
retail_df = pd.read_csv(retail_path, encoding='cp1252')

# Worked fine, so I commented out
# netflix_df.head(2)
# retail_df.head(2)

Netflix

In [ ]:
# 1. Identify columns with missing values
columns_with_null = netflix_df.columns[netflix_df.isnull().any()].tolist()
print("Columns with missing values:", columns_with_null)

# 2. Show number of missing values per column
missing_counts = netflix_df.isnull().sum()
print("\nMissing values per column:")
print(missing_counts[missing_counts > 0])

# 3. Show percentage of missing values
missing_percentages = (netflix_df.isnull().sum() / len(netflix_df)) * 100
print("\nPercentage of missing values:")
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

In [17]:
print("Netflix Dataset Shape:", netflix_df.shape)
print("\nMissing Values Analysis:")
missing_counts = netflix_df.isnull().sum()
missing_pct = (missing_counts / len(netflix_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False))

# Strategic handling based on business context
def clean_netflix_data(df):
    df_clean = df.copy()
    
    # 1. DIRECTOR (29.9% missing) - Fill with 'Unknown'
    # Why: Director is important for search/filtering, but can be unknown
    # Using 'Unknown' preserves the data for analysis without losing rows
    df_clean['director'] = df_clean['director'].fillna('Unknown')
    print("✓ Director: Filled 2,634 missing values with 'Unknown'")
    
    # 2. CAST (9.4% missing) - Fill with 'Not Available'
    # Why: Similar to director, cast matters for recommendations
    # 'Not Available' is more appropriate than 'Unknown' for cast
    df_clean['cast'] = df_clean['cast'].fillna('Not Available')
    print("✓ Cast: Filled 825 missing values with 'Not Available'")
    
    # 3. COUNTRY (9.4% missing) - Fill with most frequent country
    # Why: Country helps with regional content analysis
    # Mode imputation makes sense for categorical data
    most_common_country = df_clean['country'].mode()[0]
    df_clean['country'] = df_clean['country'].fillna(most_common_country)
    print(f"✓ Country: Filled 831 missing values with '{most_common_country}'")
    
    # 4. DATE_ADDED (0.1% missing - only 10 rows) - Drop these rows
    # Why: Very few missing (0.11%), date is important for time analysis
    # Better to drop few rows than create incorrect dates
    df_clean = df_clean.dropna(subset=['date_added'])
    print("✓ Date Added: Dropped 10 rows with missing dates")
    
    # 5. RATING (0.05% missing - only 4 rows) - Fill with most common rating
    # Why: Rating is important for content classification
    # Very few missing, mode imputation is safe
    most_common_rating = df_clean['rating'].mode()[0]
    df_clean['rating'] = df_clean['rating'].fillna(most_common_rating)
    print(f"✓ Rating: Filled 4 missing values with '{most_common_rating}'")
    
    # 6. DURATION (0.03% missing - only 3 rows) - Drop these rows
    # Why: Duration is critical for understanding content length
    # Only 3 missing, safer to drop than impute incorrectly
    df_clean = df_clean.dropna(subset=['duration'])
    print("✓ Duration: Dropped 3 rows with missing duration")
    
    return df_clean

# Clean the data
netflix_df_clean = clean_netflix_data(netflix_df)

# Verify cleaning
print("\n" + "="*50)
print("CLEANING SUMMARY")
print("="*50)
print(f"Original shape: {netflix_df.shape}")
print(f"Cleaned shape: {netflix_df_clean.shape}")
print(f"Total rows removed: {netflix_df.shape[0] - netflix_df_clean.shape[0]}")
print(f"Remaining missing values: {netflix_df_clean.isnull().sum().sum()}")

# Quality check - show sample of cleaned columns
print("\nSample of cleaned data:")
print(netflix_df_clean[['director', 'cast', 'country', 'rating']].head(3))

Netflix Dataset Shape: (8807, 12)

Missing Values Analysis:
            Missing Count  Percentage
director             2634   29.908028
country               831    9.435676
cast                  825    9.367549
date_added             10    0.113546
rating                  4    0.045418
duration                3    0.034064
✓ Director: Filled 2,634 missing values with 'Unknown'
✓ Cast: Filled 825 missing values with 'Not Available'
✓ Country: Filled 831 missing values with 'United States'
✓ Date Added: Dropped 10 rows with missing dates
✓ Rating: Filled 4 missing values with 'TV-MA'
✓ Duration: Dropped 3 rows with missing duration

CLEANING SUMMARY
Original shape: (8807, 12)
Cleaned shape: (8794, 12)
Total rows removed: 13
Remaining missing values: 0

Sample of cleaned data:
          director                                               cast  \
0  Kirsten Johnson                                      Not Available   
1          Unknown  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban..

Retail

In [ ]:
# 1. Identify columns with missing values
columns_with_null = retail_df.columns[retail_df.isnull().any()].tolist()
print("Columns with missing values:", columns_with_null)

# 2. Show number of missing values per column
missing_counts = retail_df.isnull().sum()
print("\nMissing values per column:")
print(missing_counts[missing_counts > 0])

# 3. Show percentage of missing values
missing_percentages = (retail_df.isnull().sum() / len(retail_df)) * 100
print("\nPercentage of missing values:")
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

In [18]:
print("Retail Dataset Shape:", retail_df.shape)
print("\nMissing Values Analysis:")
missing_counts = retail_df.isnull().sum()
missing_pct = (missing_counts / len(retail_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False))

def clean_retail_data(df):
    df_clean = df.copy()
    
    # 1. CUSTOMERID (24.9% missing - 135,080 rows)
    # Why: CustomerID is critical for customer analytics, RFM analysis
    # Many approaches possible - choose based on analysis goals
    print("\n⚠️ CustomerID has 24.9% missing values")
    print("Options for handling CustomerID:")
    print("  A) Drop all rows without CustomerID (loses 24.9% data)")
    print("  B) Fill with 'Unknown' (preserves transactions but limits customer analysis)")
    print("  C) Analyze transactions separately vs customer behavior")
    
    # For this example, we'll create two versions:
    # Version 1: For customer analytics (keep only customers with IDs)
    df_customer_analytics = df_clean.dropna(subset=['CustomerID'])
    print(f"\n✓ Customer Analytics version: Kept {len(df_customer_analytics)} rows ({len(df_customer_analytics)/len(df_clean)*100:.1f}%)")
    
    # Version 2: For sales/transaction analysis (fill missing CustomerID)
    df_sales_analytics = df_clean.copy()
    df_sales_analytics['CustomerID'] = df_sales_analytics['CustomerID'].fillna(0)
    print(f"✓ Sales Analytics version: Filled 135,080 missing CustomerIDs with 0")
    
    # 2. DESCRIPTION (0.27% missing - 1,454 rows)
    # Why: Description helps identify products, but only 0.27% missing
    # Could drop or fill with placeholders
    df_clean['Description'] = df_clean['Description'].fillna('Unknown')
    # Alternative: Drop rows
    # df_clean = df_clean.dropna(subset=['Description'])
    print("✓ Description: Filled 1,454 missing values with 'Unknown'")
    
    # Additional data quality checks for Retail
    # Check for negative quantities (returns)
    if 'Quantity' in df_clean.columns:
        negative_qty = df_clean[df_clean['Quantity'] < 0]
        print(f"\n⚠️ Found {len(negative_qty)} rows with negative quantity (returns)")
        # Can keep or separate for analysis
    
    # Check for zero/negative unit price
    if 'UnitPrice' in df_clean.columns:
        invalid_price = df_clean[df_clean['UnitPrice'] <= 0]
        print(f"⚠️ Found {len(invalid_price)} rows with invalid unit price")
    
    return df_clean, df_customer_analytics, df_sales_analytics

# Clean the data
retail_df_clean, retail_df_customers, retail_df_sales = clean_retail_data(retail_df)

print("\n" + "="*50)
print("CLEANING SUMMARY")
print("="*50)
print(f"Original shape: {retail_df.shape}")
print(f"Cleaned shape: {retail_df_clean.shape}")
print(f"Customer Analytics shape: {retail_df_customers.shape}")
print(f"Sales Analytics shape: {retail_df_sales.shape}")


Retail Dataset Shape: (541909, 8)

Missing Values Analysis:
             Missing Count  Percentage
CustomerID          135080   24.926694
Description           1454    0.268311

⚠️ CustomerID has 24.9% missing values
Options for handling CustomerID:
  A) Drop all rows without CustomerID (loses 24.9% data)
  B) Fill with 'Unknown' (preserves transactions but limits customer analysis)
  C) Analyze transactions separately vs customer behavior

✓ Customer Analytics version: Kept 406829 rows (75.1%)
✓ Sales Analytics version: Filled 135,080 missing CustomerIDs with 0
✓ Description: Filled 1,454 missing values with 'Unknown'

⚠️ Found 10624 rows with negative quantity (returns)
⚠️ Found 2517 rows with invalid unit price

CLEANING SUMMARY
Original shape: (541909, 8)
Cleaned shape: (541909, 8)
Customer Analytics shape: (406829, 8)
Sales Analytics shape: (541909, 8)
